In [4]:
import glob
import os
import json
import subprocess

def get_video_info(video_path):
    """Get video information using ffprobe"""
    try:
        # Get duration and fps using ffprobe
        cmd = [
            'ffprobe', 
            '-v', 'error', 
            '-select_streams', 'v:0', 
            '-show_entries', 'stream=duration,r_frame_rate', 
            '-of', 'json', 
            video_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        info = json.loads(result.stdout)
        
        # Extract duration and fps
        duration = float(info['streams'][0]['duration'])
        
        # r_frame_rate is usually in the format "num/den"
        fps_str = info['streams'][0]['r_frame_rate']
        if '/' in fps_str:
            num, den = map(float, fps_str.split('/'))
            fps = num / den
        else:
            fps = float(fps_str)
            
        return {'duration': duration, 'fps': fps}
    
    except Exception as e:
        print(f"Error getting video info: {e}")
        return {'duration': 0, 'fps': 0}

def extract_clip_ffmpeg(input_file, output_file, start_time, end_time):
    """Extract clip using ffmpeg directly"""
    try:
        # Calculate clip duration
        duration = end_time - start_time
        
        # Use ffmpeg to extract the clip
        cmd = [
            'ffmpeg',
            '-i', input_file,
            '-ss', str(start_time),
            '-t', str(duration),
            '-c:v', 'libx264',  # Video codec
            '-c:a', 'aac',      # Audio codec
            '-y',               # Overwrite output file
            output_file
        ]
        
        subprocess.run(cmd, check=True)
        return True
    
    except subprocess.CalledProcessError as e:
        print(f"FFmpeg error: {e}")
        return False
    
    except Exception as e:
        print(f"Error extracting clip: {e}")
        return False

def process_video(video_path, output_folder):
    """Process a single video file"""
    # Create output folder if it doesn't exist
    gesture_folder = os.path.join(output_folder, 'gestures')
    os.makedirs(gesture_folder, exist_ok=True)
    
    # Get the video identifier (filename without extension)
    base_filename = os.path.splitext(os.path.basename(video_path))[0].replace('.h264', '')
    
    # Extract speaker ID and gesture ID from the filename
    # Assuming filename format like "123-456.h264.mp4" where 123 is speakerID and 456 is gestureID
    try:
        filename_parts = base_filename.split('-')
        speaker_id = filename_parts[0]
        gesture_base_id = filename_parts[1] if len(filename_parts) > 1 else "unknown"
    except Exception as e:
        print(f"Error parsing filename {base_filename}: {e}")
        speaker_id = "unknown"
        gesture_base_id = "unknown"
    
    # Get the actions file path
    actions_file = video_path.replace('.h264.mp4', '.actions.json')
    
    try:
        # Get video information
        video_info = get_video_info(video_path)
        fps = video_info['fps']
        duration = video_info['duration']
        
        if fps == 0 or duration == 0:
            print(f"Could not get video info for {video_path}")
            return
            
        print(f"Video info: {duration:.2f}s, {fps:.2f}fps")
        
        # Load actions
        with open(actions_file, 'r') as f:
            actions = json.load(f)
        
        # Process gesture clips with 1 second padding
        for idx, action in enumerate(actions):
            # Calculate times with 1 second padding
            start_time = max(0, (action['start_frame'] / fps) - 1)  # Add 1 sec before, but don't go below 0
            end_time = min(duration, (action['end_frame'] / fps) + 1)  # Add 1 sec after, but don't exceed duration
            
            # Create the new filename using the requested format: ZHUBO_SPEAKERID_GESTUREID_NA
            # Where NA is the index of the gesture in the actions file
            output_file = os.path.join(gesture_folder, f'ZHUBO_{speaker_id}_{gesture_base_id}_{idx:04d}.mp4')
            
            if not os.path.exists(output_file):  # Only extract if file doesn't exist
                print(f"Extracting gesture clip {idx} from ZHUBO_{speaker_id}_{gesture_base_id}: {start_time:.2f}s - {end_time:.2f}s (with padding)")
                if extract_clip_ffmpeg(video_path, output_file, start_time, end_time):
                    print(f"Successfully extracted gesture clip: {output_file}")
            else:
                print(f"Skipping existing gesture clip: {output_file}")
                
    except Exception as e:
        print(f"Error processing video {video_path}: {e}")

def main():
    # Configuration
    data_folder = './zhubo_split_9/'
    output_folder = './for_analysis/'
    
    # Get list of videos
    video_list = glob.glob(os.path.join(data_folder, '*.h264.mp4'))
    
    # Process each video
    for video_path in video_list:
        print(f"\nProcessing video: {video_path}")
        process_video(video_path, output_folder)

if __name__ == "__main__":
    main()


Processing video: ./zhubo_split_9\9-000.h264.mp4
Video info: 88.44s, 25.00fps
Extracting gesture clip 0 from ZHUBO_9_000: 0.00s - 3.24s (with padding)
Successfully extracted gesture clip: ./for_analysis/gestures\ZHUBO_9_000_0000.mp4
Extracting gesture clip 1 from ZHUBO_9_000: 14.96s - 18.76s (with padding)
Successfully extracted gesture clip: ./for_analysis/gestures\ZHUBO_9_000_0001.mp4
Extracting gesture clip 2 from ZHUBO_9_000: 23.12s - 31.12s (with padding)
Successfully extracted gesture clip: ./for_analysis/gestures\ZHUBO_9_000_0002.mp4
Extracting gesture clip 3 from ZHUBO_9_000: 33.76s - 38.12s (with padding)
Successfully extracted gesture clip: ./for_analysis/gestures\ZHUBO_9_000_0003.mp4
Extracting gesture clip 4 from ZHUBO_9_000: 36.20s - 39.48s (with padding)
Successfully extracted gesture clip: ./for_analysis/gestures\ZHUBO_9_000_0004.mp4
Extracting gesture clip 5 from ZHUBO_9_000: 37.76s - 42.76s (with padding)
Successfully extracted gesture clip: ./for_analysis/gestures\ZH